# Robust04 Final Project Part A

This notebook (1) reproduces tuning on the 50 judged queries (301-350) and (2) generates the three submission runs for the 199 test queries (351-450 and 601-671, 673-700).


In [ ]:
from collections import defaultdict
from pathlib import Path
import sys
import zipfile

import torch

from pyserini.encode import SpladeQueryEncoder
from pyserini.search.lucene import LuceneHnswDenseSearcher, LuceneImpactSearcher, LuceneSearcher


In [ ]:
QUERIES_PATH = Path('Files-20260104/queriesROBUST.txt')
QRELS_PATH = Path('Files-20260104/qrels_50_Queries')

def read_queries_tsv(path: Path):
    queries = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        qid, query = line.split('\t', 1)
        queries[qid] = query
    return queries

def read_qrels(path: Path):
    qrels = defaultdict(dict)
    for line in path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 4:
            continue
        qid, _, docid, rel = parts
        qrels[qid][docid] = int(rel)
    return qrels

def average_precision(docids, rels):
    num_rel = sum(1 for r in rels.values() if r > 0)
    if num_rel == 0:
        return 0.0
    hit = 0
    s = 0.0
    for i, d in enumerate(docids, start=1):
        if rels.get(d, 0) > 0:
            hit += 1
            s += hit / i
    return s / num_rel

def mean_ap(run, qrels):
    return sum(average_precision(run[qid], qrels[qid]) for qid in run) / len(run)

def minmax_norm(scores_dict):
    if not scores_dict:
        return {}
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx - mn < 1e-9:
        return {d: 0.0 for d in scores_dict}
    return {d: (s - mn) / (mx - mn) for d, s in scores_dict.items()}

def fuse_weighted_minmax(runs_scores, weights, depth=1000):
    norms = [minmax_norm(rs) for rs in runs_scores]
    docs = set()
    for n in norms:
        docs |= set(n.keys())
    fused_scores = {}
    for d in docs:
        s = 0.0
        for w, n in zip(weights, norms):
            s += w * n.get(d, 0.0)
        fused_scores[d] = s
    ranked = sorted(fused_scores.items(), key=lambda x: (-x[1], x[0]))
    return [d for d, _ in ranked[:depth]]

def retrieve_run(searcher, queries, k=1000):
    run = {}
    for qid, query in queries.items():
        hits = searcher.search(query, k=k)
        run[qid] = [h.docid for h in hits]
    return run

def retrieve_scores(searcher, queries, k=1000):
    scores = {}
    for qid, query in queries.items():
        hits = searcher.search(query, k=k)
        scores[qid] = {h.docid: float(h.score) for h in hits}
    return scores

all_queries = read_queries_tsv(QUERIES_PATH)
train_qids = list(all_queries.keys())[:50]
train_queries = {qid: all_queries[qid] for qid in train_qids}
qrels = read_qrels(QRELS_PATH)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


In [ ]:
rm3 = LuceneSearcher.from_prebuilt_index('robust04')
rm3.set_bm25(0.9, 0.4)
rm3.set_rm3(20, 5, 0.5)

spladepp_encoder = SpladeQueryEncoder('naver/splade-cocondenser-ensembledistil', device=device)
spladepp = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-pp-ed', spladepp_encoder)

spladev3_encoder = SpladeQueryEncoder('naver/splade-v3-distilbert', device=device)
spladev3 = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-v3', spladev3_encoder)

dense = LuceneHnswDenseSearcher.from_prebuilt_index(
    'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw',
    ef_search=1000,
    encoder='BgeBaseEn15',
)

rm3_run = retrieve_run(rm3, train_queries)
spladepp_run = retrieve_run(spladepp, train_queries)
spladev3_run = retrieve_run(spladev3, train_queries)
dense_run = retrieve_run(dense, train_queries)

print('RM3 MAP:', f'{mean_ap(rm3_run, qrels):.4f}')
print('SPLADE++ MAP:', f'{mean_ap(spladepp_run, qrels):.4f}')
print('SPLADE-v3-distil MAP:', f'{mean_ap(spladev3_run, qrels):.4f}')
print('Dense (BGE) MAP:', f'{mean_ap(dense_run, qrels):.4f}')

rm3_scores = retrieve_scores(rm3, train_queries)
spladepp_scores = retrieve_scores(spladepp, train_queries)
spladev3_scores = retrieve_scores(spladev3, train_queries)
dense_scores = retrieve_scores(dense, train_queries)

run_2_weights = (0.60, 0.25, 0.15)
run_3_weights = (0.55, 0.10, 0.15, 0.20)

run2 = {
    qid: fuse_weighted_minmax([rm3_scores[qid], spladepp_scores[qid], dense_scores[qid]], run_2_weights)
    for qid in train_queries
}
run3 = {
    qid: fuse_weighted_minmax([rm3_scores[qid], spladepp_scores[qid], spladev3_scores[qid], dense_scores[qid]], run_3_weights)
    for qid in train_queries
}

print('Fusion run_2 MAP:', f'{mean_ap(run2, qrels):.4f}')
print('Fusion run_3 MAP:', f'{mean_ap(run3, qrels):.4f}')


## Generate submission runs

This uses `generate_runs.py` to write `run_1.res`, `run_2.res`, `run_3.res` for the 199 test queries.


In [ ]:
import subprocess

subprocess.run([
    sys.executable,
    'generate_runs.py',
    '--out1', 'run_1.res',
    '--out2', 'run_2.res',
    '--out3', 'run_3.res',
], check=True)


In [ ]:
EXPECTED_QIDS = set(map(str, list(range(351, 451)) + list(range(601, 672)) + list(range(673, 701))))

def sanity_check_run(path):
    qids = set()
    bad = 0
    line_count = 0
    last_qid = None
    last_rank = 0
    last_score = None
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            line_count += 1
            parts = line.strip().split()
            if len(parts) != 6:
                bad += 1
                continue
            qid, q0, docid, rank, score, tag = parts
            qids.add(qid)
            if q0 != 'Q0':
                bad += 1
            r = int(rank)
            s = float(score)
            if last_qid != qid:
                last_qid = qid
                last_rank = 0
                last_score = None
            if r != last_rank + 1:
                bad += 1
            if last_score is not None and s > last_score + 1e-6:
                bad += 1
            last_rank = r
            last_score = s
    missing = EXPECTED_QIDS - qids
    extra = qids - EXPECTED_QIDS
    return {
        'lines': line_count,
        'unique_qids': len(qids),
        'missing_qids': len(missing),
        'extra_qids': len(extra),
        'bad_checks': bad,
    }

for fname in ['run_1.res', 'run_2.res', 'run_3.res']:
    print(fname, sanity_check_run(fname))


In [ ]:
zip_name = 'Final_Project_Part_A_runs.zip'
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for fname in ['run_1.res', 'run_2.res', 'run_3.res']:
        z.write(fname, arcname=fname)
zip_name
